# Fabric LakeDB + Delta-RS Example (Spark-Free)

This notebook demonstrates **Spark-free Delta Lake operations** with **Microsoft Fabric LakeDB** using LakeLogic and Delta-RS.

## 🚀 Features

- ✅ Read Fabric LakeDB tables (no Spark!)
- ✅ Validate data with LakeLogic contracts
- ✅ Atomic MERGE operations (upsert)
- ✅ Calculated fields & transformations
- ✅ Time travel (version/timestamp)
- ✅ Azure AD authentication

## 📋 Prerequisites

```bash
pip install "lakelogic[delta]"
pip install azure-identity  # For Azure AD auth
```

## Setup: Configure Credentials

In [ ]:
import os
from lakelogic import DataProcessor
from lakelogic.engines.delta_adapter import DeltaAdapter
from lakelogic.engines.unity_catalog import resolve_catalog_path
import polars as pl

# Option 1: Account Key Authentication
os.environ["AZURE_STORAGE_ACCOUNT_NAME"] = "onelake"
os.environ["AZURE_STORAGE_ACCOUNT_KEY"] = "..."

# Option 2: Azure AD Authentication (Recommended)
# Run: az login
# Credentials are automatically picked up

print("✅ Credentials configured")

## Example 1: Read Fabric LakeDB Table

Use Fabric table names directly (`workspace.lakehouse.table`) - LakeLogic automatically resolves them to OneLake paths!

In [ ]:
# Create processor with Polars engine (no Spark!)
processor = DataProcessor(
    engine="polars",
    contract="fabric_lakedb_contract.yaml"
)

# Read Fabric LakeDB table
good_df, bad_df = processor.run_source("myworkspace.sales_lakehouse.transactions")

print(f"✅ Good records: {len(good_df)}")
print(f"❌ Quarantined records: {len(bad_df)}")

# Display good data
good_df.head()

In [ ]:
# Display quarantined data (if any)
if len(bad_df) > 0:
    print("Quarantined records:")
    bad_df.head()
else:
    print("No quarantined records! 🎉")

## Example 2: Table Name Resolution

See how Fabric table names are automatically resolved to OneLake paths.

In [ ]:
# Resolve Fabric table name
table_name = "myworkspace.sales_lakehouse.transactions"
storage_path = resolve_catalog_path(table_name, platform="fabric")

print(f"Table name: {table_name}")
print(f"OneLake path: {storage_path}")
print(f"\n✅ LakeLogic automatically handles this resolution!")

## Example 3: MERGE Operation (Upsert)

Perform atomic MERGE operations **without Spark**!

In [ ]:
# Create new/updated transaction data
new_transactions = pl.DataFrame({
    "transaction_id": ["TXN001", "TXN002", "TXN999"],
    "customer_id": [101, 102, 999],
    "product_id": ["PROD-A", "PROD-B", "PROD-C"],
    "quantity": [2, 5, 1],
    "unit_price": [29.99, 49.99, 19.99],
    "total_amount": [59.98, 249.95, 19.99],
    "transaction_date": ["2026-02-09", "2026-02-09", "2026-02-09"],
    "payment_method": ["credit_card", "paypal", "debit_card"]
})

print("New/updated transactions:")
new_transactions

In [ ]:
# MERGE into Fabric LakeDB table (atomic, no Spark!)
adapter = DeltaAdapter()
stats = adapter.merge(
    target_path="myworkspace.sales_lakehouse.transactions",
    source_df=new_transactions,
    merge_key="transaction_id"
)

print(f"✅ MERGE complete:")
print(f"  - Updated: {stats['num_updated']} records")
print(f"  - Inserted: {stats['num_inserted']} records")

## Example 4: Calculated Fields

LakeLogic automatically calculates missing fields using contract transformations.

In [ ]:
# Create transaction data with missing total_amount
transactions_with_missing = pl.DataFrame({
    "transaction_id": ["TXN100", "TXN101"],
    "customer_id": [201, 202],
    "product_id": ["PROD-X", "PROD-Y"],
    "quantity": [3, 2],
    "unit_price": [15.00, 25.00],
    "total_amount": [None, None],  # Missing - will be calculated
    "transaction_date": ["2026-02-09", "2026-02-09"],
    "payment_method": ["cash", "credit_card"]
})

print("Before processing (missing total_amount):")
transactions_with_missing

In [ ]:
# Process with contract (calculates total_amount)
processor = DataProcessor(
    engine="polars",
    contract="fabric_lakedb_contract.yaml"
)
good_df, bad_df = processor.run(transactions_with_missing)

print("After processing (total_amount calculated):")
good_df[["transaction_id", "quantity", "unit_price", "total_amount"]]

## Example 5: Sales Analytics

Analyze sales data directly from Fabric LakeDB.

In [ ]:
# Read transaction data
df = adapter.read("myworkspace.sales_lakehouse.transactions")

# Calculate total sales by payment method
sales_by_payment = df.group_by("payment_method").agg([
    pl.count().alias("transaction_count"),
    pl.sum("total_amount").alias("total_sales"),
    pl.mean("total_amount").alias("avg_transaction")
]).sort("total_sales", descending=True)

print("Sales by Payment Method:")
sales_by_payment

In [ ]:
# Calculate daily sales
daily_sales = df.group_by("transaction_date").agg([
    pl.count().alias("transactions"),
    pl.sum("total_amount").alias("total_sales")
]).sort("transaction_date")

print("Daily Sales:")
daily_sales

## Example 6: Time Travel

Access historical versions of your Delta tables.

In [ ]:
# Read specific version
df_v1 = adapter.read("myworkspace.sales_lakehouse.transactions", version=1)
print(f"Version 1: {len(df_v1)} records")
df_v1.head()

In [ ]:
# Get table history
history = adapter.get_history("myworkspace.sales_lakehouse.transactions", limit=10)
print("Table history (last 10 commits):")
history

## Example 7: Complete Pipeline

Read → Validate → MERGE in one complete workflow.

In [ ]:
# Step 1: Read from Fabric LakeDB
print("Step 1: Reading from Fabric LakeDB...")
processor = DataProcessor(
    engine="polars",
    contract="fabric_lakedb_contract.yaml"
)
good_df, bad_df = processor.run_source("myworkspace.sales_lakehouse.transactions")
print(f"  ✅ Good: {len(good_df)}, ❌ Bad: {len(bad_df)}")

# Step 2: MERGE validated data
print("\nStep 2: MERGE validated data...")
adapter = DeltaAdapter()
stats = adapter.merge(
    target_path="myworkspace.sales_lakehouse.transactions_validated",
    source_df=good_df,
    merge_key="transaction_id"
)
print(f"  ✅ Updated: {stats['num_updated']}, Inserted: {stats['num_inserted']}")

# Step 3: Write quarantined data
if len(bad_df) > 0:
    print("\nStep 3: Writing quarantined data...")
    adapter.write(
        df=bad_df,
        path="myworkspace.sales_lakehouse.transactions_quarantine",
        mode="append"
    )
    print(f"  ✅ Wrote {len(bad_df)} quarantined records")

print("\n✅ Pipeline complete!")

## Example 8: Azure AD Authentication (Recommended)

Use Azure AD for more secure authentication.

In [ ]:
from azure.identity import DefaultAzureCredential

# Get Azure AD token
credential = DefaultAzureCredential()
token = credential.get_token("https://storage.azure.com/.default")

# Create adapter with Azure AD auth
adapter_ad = DeltaAdapter(storage_options={
    "AZURE_STORAGE_ACCOUNT_NAME": "onelake",
    "BEARER_TOKEN": token.token
})

# Read with Azure AD auth
df = adapter_ad.read("myworkspace.sales_lakehouse.transactions")
print(f"✅ Read {len(df)} records using Azure AD authentication")

## 🎯 Summary

### What We Demonstrated:

✅ **Fabric LakeDB table names** - Use `workspace.lakehouse.table` directly  
✅ **Spark-free Delta Lake** - Read/write with Polars (10-100x faster)  
✅ **Atomic MERGE** - Upsert operations without Spark  
✅ **Calculated fields** - Automatic transformations  
✅ **Sales analytics** - Direct querying with Polars  
✅ **Time travel** - Access historical versions  
✅ **Azure AD auth** - Secure authentication  

### Learn More:

- 📚 [Delta Lake Support](../../docs/delta_lake_support.md)
- 📚 [Catalog Table Names](../../docs/catalog_table_names.md)
- 📚 [Fabric LakeDB Contract](fabric_lakedb_contract.yaml)

---

*Last Updated: February 2026*